# Bootstrap Sharpe Analysis

This notebook evaluates the statistical uncertainty around the Sharpe ratios obtained from the backtest.

A block bootstrap is applied to portfolio excess returns in order to preserve short-term time dependence while generating alternative return samples.

The analysis reports:

- 95% confidence intervals for each strategy's Sharpe ratio
- 95% confidence intervals for pairwise Sharpe differences
- results for both raw and volatility-targeted portfolios

In [13]:
import sys
sys.path.append("../src")

import importlib
import pandas as pd
from itertools import combinations

import metrics
import backtesting
import bootstrap

importlib.reload(metrics)
importlib.reload(backtesting)
importlib.reload(bootstrap)

WINDOW = 3 * 252
STEP = 21
TRANSACTION_COST = 0.001
VOL_TARGET = 0.10
MAX_LEVERAGE = 2.0
FUNDING_SPREAD = 0.01

BLOCK_SIZE = 21
N_BOOTSTRAP = 10_000
SEED = 42

## Backtest returns

The same backtest configuration used in the main analysis is reproduced here.

Sharpe ratios are bootstrapped from **excess returns** rather than total returns. The risk-free return is therefore removed before resampling.

In [14]:
prices = pd.read_csv("../data/etf_prices_raw.csv", index_col="Date", parse_dates=True)
returns = metrics.compute_returns(prices)

rf_rate = pd.read_csv("../data/risk_free_rate.csv", index_col="Date", parse_dates=True)["risk_free_rate"]
rf_rate = pd.to_numeric(rf_rate, errors="coerce").ffill()

annual_rf = rf_rate / 100
risk_free_returns = (1 + annual_rf) ** (1 / 252) - 1
risk_free_returns = risk_free_returns.reindex(returns.index).ffill()

groups = {
    "equities": ["SPY", "QQQ", "EWU", "EZU", "EEM"],
    "bonds": ["IEF", "SHY"],
    "alternatives": ["GLD", "VNQ"]
}

strategies = {
    "Equal Weight": (backtesting.equal_strategy, None),
    "Inverse Vol": (backtesting.risk_strategy, None),
    "Max Geo": (backtesting.max_geo_strategy, None),
    "Max Sharpe": (backtesting.max_sharpe_strategy, None),
    "Carver": (backtesting.carver_handcrafting_strategy, groups),
    "Hierarchical Inv Vol": (backtesting.inv_handcrafting_strategy, groups),
    "SPY": (backtesting.spy_strategy, None),
    "60/40": (backtesting.sixty_forty_strategy, None),
    "Tech": (backtesting.tech_strategy, None)
}

raw_portfolio_returns = {}

for name, (strategy, strategy_groups) in strategies.items():
    port_ret, _, _, _, _ = backtesting.backtest(
        returns,
        window=WINDOW,
        step=STEP,
        allocation_func=strategy,
        groups=strategy_groups,
        vol_target=None,
        risk_free_returns=risk_free_returns,
        transaction_cost=TRANSACTION_COST,
        funding_spread=FUNDING_SPREAD
    )

    raw_portfolio_returns[name] = port_ret

targeted_portfolio_returns = {}

for name, (strategy, strategy_groups) in strategies.items():
    port_ret, _, _, _, _ = backtesting.backtest(
        returns,
        window=WINDOW,
        step=STEP,
        allocation_func=strategy,
        groups=strategy_groups,
        vol_target=VOL_TARGET,
        max_leverage=MAX_LEVERAGE,
        risk_free_returns=risk_free_returns,
        transaction_cost=TRANSACTION_COST,
        funding_spread=FUNDING_SPREAD
    )

    targeted_portfolio_returns[name] = port_ret

raw_returns_ts = pd.DataFrame(raw_portfolio_returns)
targeted_returns_ts = pd.DataFrame(targeted_portfolio_returns)

raw_excess = bootstrap.make_excess_returns(raw_returns_ts, risk_free_returns)
targeted_excess = bootstrap.make_excess_returns(targeted_returns_ts, risk_free_returns)

## Raw portfolios

The block bootstrap uses a **21-trading-day block size** and **10,000 bootstrap samples**.

The first table reports the 95% confidence interval of each strategy's Sharpe ratio.  
The second table reports confidence intervals for pairwise Sharpe differences.

In [15]:
results_raw = bootstrap.bootstrap_sharpe(
    raw_excess,
    block_size=BLOCK_SIZE,
    n_bootstrap=N_BOOTSTRAP,
    seed=SEED
)

confidence_intervals_raw = pd.DataFrame({
    "CI Lower": results_raw.quantile(0.025),
    "CI Upper": results_raw.quantile(0.975)
})

comparisons_raw = []

for strat1, strat2 in combinations(results_raw.columns, 2):
    diff = results_raw[strat1] - results_raw[strat2]

    comparisons_raw.append({
        "Strategy 1": strat1,
        "Strategy 2": strat2,
        "Mean Diff": diff.mean(),
        "CI Lower": diff.quantile(0.025),
        "CI Upper": diff.quantile(0.975)
    })

comparison_table_raw = pd.DataFrame(comparisons_raw)

display(
    confidence_intervals_raw.style
    .format({
        "CI Lower": "{:.3f}",
        "CI Upper": "{:.3f}"
    })
    .set_caption("Raw Portfolios — 95% Bootstrap Sharpe Confidence Intervals")
)

display(
    comparison_table_raw.style
    .format({
        "Mean Diff": "{:.3f}",
        "CI Lower": "{:.3f}",
        "CI Upper": "{:.3f}"
    })
    .set_caption("Raw Portfolios — Pairwise Sharpe Differences")
)

,CI Lower,CI Upper
Equal Weight,0.068,0.903
Inverse Vol,0.060,0.961
Max Geo,0.240,1.102
Max Sharpe,0.187,1.009
Carver,0.074,0.984
Hierarchical Inv Vol,0.075,0.987
SPY,0.190,1.023
60/40,0.230,1.114
Tech,0.327,1.185


,Strategy 1,Strategy 2,Mean Diff,CI Lower,CI Upper
0,Equal Weight,Inverse Vol,-0.027,-0.174,0.125
1,Equal Weight,Max Geo,-0.185,-0.434,0.071
2,Equal Weight,Max Sharpe,-0.114,-0.551,0.306
3,Equal Weight,Carver,-0.053,-0.339,0.239
4,Equal Weight,Hierarchical Inv Vol,-0.053,-0.326,0.226
5,Equal Weight,SPY,-0.116,-0.288,0.055
6,Equal Weight,60/40,-0.184,-0.349,-0.021
7,Equal Weight,Tech,-0.268,-0.507,-0.026
8,Inverse Vol,Max Geo,-0.158,-0.440,0.122
9,Inverse Vol,Max Sharpe,-0.087,-0.495,0.294


## Volatility-targeted portfolios

The same bootstrap procedure is applied to the volatility-targeted portfolio excess returns using the same block size, number of bootstrap samples and random seed.

In [17]:
results_targeted = bootstrap.bootstrap_sharpe(
    targeted_excess,
    block_size=BLOCK_SIZE,
    n_bootstrap=N_BOOTSTRAP,
    seed=SEED
)

confidence_intervals_targeted = pd.DataFrame({
    "CI Lower": results_targeted.quantile(0.025),
    "CI Upper": results_targeted.quantile(0.975)
})

comparisons_targeted = []

for strat1, strat2 in combinations(results_targeted.columns, 2):
    diff = results_targeted[strat1] - results_targeted[strat2]

    comparisons_targeted.append({
        "Strategy 1": strat1,
        "Strategy 2": strat2,
        "Mean Diff": diff.mean(),
        "CI Lower": diff.quantile(0.025),
        "CI Upper": diff.quantile(0.975)
    })

comparison_table_targeted = pd.DataFrame(comparisons_targeted)

display(
    confidence_intervals_targeted.style
    .format({
        "CI Lower": "{:.3f}",
        "CI Upper": "{:.3f}"
    })
    .set_caption("Volatility-Targeted Portfolios — 95% Bootstrap Sharpe Confidence Intervals")
)

display(
    comparison_table_targeted.style
    .format({
        "Mean Diff": "{:.3f}",
        "CI Lower": "{:.3f}",
        "CI Upper": "{:.3f}"
    })
    .set_caption("Volatility-Targeted Portfolios — Pairwise Sharpe Differences")
)

,CI Lower,CI Upper
Equal Weight,-0.020,0.885
Inverse Vol,-0.093,0.809
Max Geo,0.228,1.109
Max Sharpe,0.230,1.073
Carver,-0.072,0.834
Hierarchical Inv Vol,-0.066,0.842
SPY,0.114,0.993
60/40,0.157,1.078
Tech,0.282,1.162


,Strategy 1,Strategy 2,Mean Diff,CI Lower,CI Upper
0,Equal Weight,Inverse Vol,0.068,-0.117,0.253
1,Equal Weight,Max Geo,-0.241,-0.447,-0.021
2,Equal Weight,Max Sharpe,-0.228,-0.605,0.147
3,Equal Weight,Carver,0.039,-0.279,0.364
4,Equal Weight,Hierarchical Inv Vol,0.031,-0.273,0.339
5,Equal Weight,SPY,-0.117,-0.305,0.070
6,Equal Weight,60/40,-0.181,-0.350,-0.010
7,Equal Weight,Tech,-0.291,-0.545,-0.033
8,Inverse Vol,Max Geo,-0.309,-0.593,-0.026
9,Inverse Vol,Max Sharpe,-0.296,-0.659,0.061
